In [ ]:
!pip install polars pandas numpy scikit-learn lightgbm catboost implicit
!pip install sentence-transformers transformers torch librosa
!pip install datasets scipy matplotlib seaborn
!pip install faiss-cpu tqdm psutil

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
from datasets import load_dataset
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer
import librosa
import torch
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def load_yambda_50m_kaggle(
    data_dir: str = "/kaggle/input/datasets/ploshkin/yambda",
    sample_fraction: float = 0.1,
    n_emb_components: int = 64,
    cache: bool = True,
    random_state: int = 42
):
    import polars as pl
    import numpy as np
    from pathlib import Path
    from sklearn.decomposition import PCA
    import gc
    import warnings
    warnings.filterwarnings("ignore")
    
    data_path = Path(data_dir)
    cache_dir = Path("/kaggle/working/yambda_cache")
    
    if cache:
        cache_dir.mkdir(exist_ok=True)
        cache_pos = cache_dir / f"pos_{int(sample_fraction*100)}pct.parquet"
        cache_emb = cache_dir / f"emb_{n_emb_components}d.parquet"
        
        if cache_pos.exists() and cache_emb.exists():
            df_pos = pl.read_parquet(cache_pos)
            df_emb = pl.read_parquet(cache_emb)
            return df_pos, df_emb
        
    df_raw = pl.read_parquet("/kaggle/input/datasets/ploshkin/yambda/flat/50m/multi_event.parquet")
    
    df_yambda_pos = df_raw.filter(
        (pl.col("event_type") == "like") | 
        ((pl.col("event_type") == "listen") & (pl.col("played_ratio_pct") >= 70))
    ).select(["uid", "item_id", "timestamp"]).unique()
    
    if sample_fraction < 1.0:
        n_samples = int(len(df_yambda_pos) * sample_fraction)
        df_yambda_pos = df_yambda_pos.sample(n=n_samples, seed=random_state)
    
    df_yambda_pos = df_yambda_pos.sort("timestamp")
    
    del df_raw
    gc.collect()

    df_emb_raw = pl.read_parquet(
        "/kaggle/input/datasets/ploshkin/yambda/embeddings.parquet",
        columns=["item_id", "embed"]
    )
    
    target_items = set(df_yambda_pos["item_id"].to_list())
    
    df_emb_filtered = df_emb_raw.filter(pl.col("item_id").is_in(target_items))
    
    embed_list = df_emb_filtered["embed"].to_list()
    embed_matrix = np.vstack(embed_list).astype(np.float32)
    item_ids = df_emb_filtered["item_id"].to_list()
    
    pca = PCA(n_components=n_emb_components, random_state=random_state)
    embed_reduced = pca.fit_transform(embed_matrix).astype(np.float32)
    
    del embed_matrix, embed_list, df_emb_raw, df_emb_filtered
    gc.collect()
    
    df_yambda_emb = pl.DataFrame(
        embed_reduced,
        schema=[f"emb_{i}" for i in range(n_emb_components)],
        schema_overrides={f"emb_{i}": pl.Float32 for i in range(n_emb_components)}
    ).with_columns(pl.Series("item_id", item_ids))
    
    del embed_reduced, item_ids
    gc.collect()
    
    if cache:
        df_yambda_pos.write_parquet(cache_pos, compression="zstd")
        df_yambda_emb.write_parquet(cache_emb, compression="zstd")
    
    print("YAMBA-50M ЗАГРУЖЕНА")
    print(f"   Взаимодействия: {len(df_yambda_pos):,}")
    print(f"   Пользователей: {df_yambda_pos['uid'].n_unique():,}")
    print(f"   Треков: {df_yambda_pos['item_id'].n_unique():,}")
    print(f"   Эмбеддингов: {len(df_yambda_emb):,} × {n_emb_components}D")
    print(f"   RAM использовано: ~2-4 ГБ")
    
    return df_yambda_pos, df_yambda_emb

def load_nto_books_safe(data_dir: str = "/kaggle/input/datasets/biwanun1690/nto-2-1511"):
    import pandas as pd
    import polars as pl
    from pathlib import Path

    data_path = Path(data_dir)

    def read_csv_robust(filename: str) -> pl.DataFrame:
        filepath = data_path / filename
        if not filepath.exists():
            return pl.DataFrame()

        try:
            with open(filepath, 'r', encoding='utf-8-sig') as f:
                header_line = f.readline().strip()

            sep = ';' if ';' in header_line else ','

            df_pd = pd.read_csv(
                filepath,
                sep=sep,
                on_bad_lines='skip',
                encoding='utf-8-sig',
                dtype=str
            )

            df_pd.columns = df_pd.columns.str.strip().str.replace('\r', '').str.replace('\n', '')

            numeric_cols = ['user_id', 'book_id', 'has_read', 'rating', 'publication_year', 'age', 'genre_id']
            for col in numeric_cols:
                if col in df_pd.columns:
                    df_pd[col] = pd.to_numeric(df_pd[col], errors='coerce')

            return pl.from_pandas(df_pd)

        except Exception as e:
            return pl.DataFrame()

    train_pl  = read_csv_robust("train.csv")
    books_pl  = read_csv_robust("books.csv")
    users_pl  = read_csv_robust("users.csv")
    desc_pl   = read_csv_robust("book_descriptions.csv")

    if "has_read" in train_pl.columns:
        df_books = train_pl.filter(
            (pl.col("has_read") == 1) &
            pl.col("user_id").is_not_null() &
            pl.col("book_id").is_not_null()
        ).select(["user_id", "book_id", "rating", "timestamp"])
    else:
        df_books = train_pl

    if "timestamp" in df_books.columns:
        df_books = df_books.with_columns(
            pl.col("timestamp").str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False).alias("dt")
        )

    print(f"   train: {len(df_books):,} позитивных записей")
    print(f"   books: {len(books_pl):,} книг")
    print(f"   users: {len(users_pl):,} пользователей")
    print(f"   desc: {len(desc_pl):,} описаний")

    return df_books, books_pl, users_pl, desc_pl

def load_musi_scene():
    try:
        musi_data = load_dataset("tina2900/musi-scene", split="train")
        df_musi = pl.from_pandas(musi_data.to_pandas())

        print(f"   ✅ Musi-Scene: {len(df_musi):,} пар трек-видео")
        print(f"   📋 Колонки: {df_musi.columns}")

        return df_musi

    except Exception as e:
        print(f"   Не удалось загрузить musi-scene: {e}")

        n_pairs = 10000
        df_musi = pl.DataFrame({
            "track_id": np.random.randint(1, 10000, n_pairs),
            "video_id": np.random.randint(1, 5000, n_pairs),
            "audio_similarity": np.random.random(n_pairs),
            "video_category": np.random.choice(["action", "drama", "comedy", "horror"], n_pairs)
        })

        return df_musi

In [ ]:
df_books, books_pl, users_pl, desc_pl = load_nto_books_safe('/kaggle/input/datasets/biwanun1690/nto-2-1511') # df_books, df_books_meta, df_users, df_desc
df_musi_scene = load_musi_scene()
df_yambda, df_yambda_emb = load_yambda_50m_kaggle(
    data_dir="/kaggle/input/ploshkin-yambda",
    sample_fraction=0.1,
    n_emb_components=64,
    cache=True
)

In [ ]:
def extract_behavioral_features(df_inter: pl.DataFrame, user_col: str = "uid", item_col: str = "item_id") -> pl.DataFrame:    
    df = df_inter.clone()
    
    if "timestamp" in df.columns:
        dtype = df["timestamp"].dtype
        if dtype in [pl.String, pl.Utf8]:
            df = df.with_columns(
                pl.col("timestamp").str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False)
                .fill_null(pl.col("timestamp").str.to_datetime("%Y-%m-%d", strict=False))
                .alias("ts_dt")
            )
        elif dtype == pl.Datetime:
            df = df.with_columns(pl.col("timestamp").alias("ts_dt"))
        elif dtype.is_integer() or dtype == pl.Float64:
            df = df.with_columns(pl.from_epoch(pl.col("timestamp"), time_unit="s").alias("ts_dt"))
        else:
            df = df.with_columns(pl.lit(None, dtype=pl.Datetime).alias("ts_dt"))
    else:
        df = df.with_columns(pl.lit(None, dtype=pl.Datetime).alias("ts_dt"))

    agg_exprs = [
        pl.count().alias("total_interactions"),
        pl.col(item_col).n_unique().alias("unique_items"),
    ]
    
    if "rating" in df.columns:
        agg_exprs.extend([
            pl.col("rating").mean().alias("avg_rating"),
            pl.col("rating").std().alias("rating_std"),
        ])
    else:
        agg_exprs.extend([pl.lit(0.0).alias("avg_rating"), pl.lit(0.0).alias("rating_std")])
        
    if "ts_dt" in df.columns:
        agg_exprs.append(
            (pl.col("ts_dt").max() - pl.col("ts_dt").min()).dt.total_seconds().alias("activity_span_seconds")
        )
    else:
        agg_exprs.append(pl.lit(0.0).alias("activity_span_seconds"))
        
    user_features = df.group_by(user_col).agg(agg_exprs)
    
    user_features = user_features.with_columns([
        (pl.col("activity_span_seconds") / 86400).alias("activity_span_days"),
    ])
    
    user_features = user_features.with_columns([
        (pl.col("total_interactions") / (pl.col("activity_span_days") + 1)).alias("interaction_density"),
        (pl.col("unique_items") / pl.col("total_interactions").clip(1)).alias("diversity_ratio"),
        (pl.col("activity_span_seconds") / (pl.col("total_interactions").clip(1) * 3600)).alias("avg_interval_hours"),
    ]).drop("activity_span_seconds")
    
    return user_features

user_feats_music = extract_behavioral_features(df_yambda, user_col="uid", item_col="item_id")

user_feats_books = extract_behavioral_features(df_books, user_col="user_id", item_col="book_id")

print(f"   Музыка: {len(user_feats_music)} пользователей")
print(f"   Книги: {len(user_feats_books)} пользователей")

In [ ]:
def extract_temporal_features(df_inter: pl.DataFrame, user_col: str) -> pl.DataFrame:    
    df = df_inter.clone()
    
    if "timestamp" in df.columns:
        dtype = df["timestamp"].dtype
        
        if dtype.is_integer() or dtype == pl.Float64:
            df = df.with_columns(pl.from_epoch(pl.col("timestamp"), time_unit="s").alias("ts_dt"))
            
        elif dtype in [pl.String, pl.Utf8]:
            df = df.with_columns(
                pl.col("timestamp").str.to_datetime("%Y-%m-%d %H:%M:%S", strict=False).alias("ts_dt")
            )
            
        elif dtype == pl.Datetime:
            df = df.with_columns(pl.col("timestamp").alias("ts_dt"))
            
        else:
            df = df.with_columns(pl.lit(None, dtype=pl.Datetime).alias("ts_dt"))
    else:
        df = df.with_columns(pl.lit(None, dtype=pl.Datetime).alias("ts_dt"))
        
    temporal = df.with_columns([
        pl.col("ts_dt").dt.hour().alias("hour"),
        pl.col("ts_dt").dt.weekday().alias("day_of_week"),  # 1=Пн, 7=Вс
        pl.col("ts_dt").dt.month().alias("month"),
    ]).group_by(user_col).agg([
        pl.col("hour").mean().alias("avg_hour"),
        pl.col("day_of_week").mean().alias("avg_day"),
        pl.col("hour").std().alias("hour_variability"),
        (pl.col("hour").filter(pl.col("hour") < 12).count() / pl.count()).alias("morning_ratio"),
        (pl.col("hour").filter((pl.col("hour") >= 12) & (pl.col("hour") < 18)).count() / pl.count()).alias("afternoon_ratio"),
        (pl.col("hour").filter(pl.col("hour") >= 18).count() / pl.count()).alias("evening_ratio"),
    ])
    
    return temporal

temporal_music = extract_temporal_features(df_yambda, "uid")
temporal_books = extract_temporal_features(df_books, "user_id")

print(f"   Музыка: {len(temporal_music)} пользователей")
print(f"   Книги: {len(temporal_books)} пользователей")

In [ ]:
def extract_content_preferences(df_inter: pl.DataFrame, df_emb: pl.DataFrame,
                                item_col: str = "item_id", user_col: str = "uid",
                                n_components: int = 32) -> pl.DataFrame:

    from sklearn.decomposition import PCA

    n_sample = min(len(df_emb), 50000)
    df_emb_sample = df_emb.sample(n=n_sample, seed=42) if n_sample < len(df_emb) else df_emb


    if "embed" in df_emb_sample.columns:
        emb_matrix = np.vstack(df_emb_sample["embed"].to_list()).astype(np.float32)
    else:
        emb_cols = [c for c in df_emb_sample.columns if c.startswith("emb_")]
        if not emb_cols:
            raise ValueError(f"Не найдены колонки эмбеддингов (emb_*) в df_emb. Доступные: {list(df_emb_sample.columns)}")
        emb_matrix = df_emb_sample[emb_cols].to_numpy().astype(np.float32)

    current_dim = emb_matrix.shape[1]
    if current_dim > n_components:
        pca = PCA(n_components=n_components, random_state=42)
        emb_reduced = pca.fit_transform(emb_matrix).astype(np.float32)
    else:
        emb_reduced = emb_matrix

    final_dim = emb_reduced.shape[1]

    
    df_emb_reduced = pl.DataFrame(
        emb_reduced,
        schema=[f"emb_{i}" for i in range(final_dim)],
        schema_overrides={f"emb_{i}": pl.Float32 for i in range(final_dim)}
    ).with_columns(pl.Series(item_col, df_emb_sample[item_col].to_list()))

    user_content = (df_inter
        .join(df_emb_reduced, on=item_col, how="left")
        .group_by(user_col)
        .agg([pl.col(f"emb_{i}").mean().alias(f"user_pref_{i}") for i in range(final_dim)])
    )

    return user_content

def create_book_bert_embeddings(descriptions: pl.DataFrame, books: pl.DataFrame,
                                sample_size: int = 5000, n_components: int = 32) -> pl.DataFrame:
    
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError:
        print(" Установите: pip install sentence-transformers")
        return None

    text_df = (descriptions
        .join(books.select(["book_id", "title", "author_name"]), on="book_id", how="left")
        .with_columns([
            pl.col("description").fill_null("").alias("desc"),
            pl.col("title").fill_null("").alias("title"),
            pl.col("author_name").fill_null("").alias("author"),
        ])
        .with_columns(
            (pl.col("title") + " " + pl.col("author") + " " + pl.col("desc"))
            .str.replace_all(r"[^\w\sа-яА-ЯёЁ\-]", " ").alias("full_text")
        )
    )
    
    if sample_size and sample_size < len(text_df):
        text_df = text_df.sample(n=sample_size, seed=42)
        
    texts = text_df["full_text"].to_list()
    book_ids = text_df["book_id"].to_list()
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer("DeepPavlov/rubert-base-cased", device=device)
    embeddings = model.encode(texts, batch_size=32, show_progress_bar=True, convert_to_numpy=True)
    
    from sklearn.decomposition import PCA
    pca = PCA(n_components=n_components, random_state=42)
    emb_reduced = pca.fit_transform(embeddings).astype(np.float32)
    
    df_emb = pl.DataFrame(
        emb_reduced,
        schema=[f"emb_{i}" for i in range(n_components)],
        schema_overrides={f"emb_{i}": pl.Float32 for i in range(n_components)}
    ).with_columns(pl.Series("book_id", book_ids))
    
    return df_emb


content_music = extract_content_preferences(df_yambda, df_yambda_emb,
                                            item_col="item_id", user_col="uid", n_components=32)

df_book_emb = create_book_bert_embeddings(desc_pl, books_pl, sample_size=5000, n_components=32)

content_books = extract_content_preferences(df_books, df_book_emb,
                                           item_col="book_id", user_col="user_id", n_components=32)

print(f"   Музыка: {len(content_music)} пользователей")
print(f"   Книги: {len(content_books)} пользователей")

In [ ]:
def perform_kmeans_clustering(user_features: pl.DataFrame,
                              n_clusters: int = 10,
                              feature_cols: list = None) -> pl.DataFrame:

    if feature_cols is None:
        exclude = ["uid", "user_id"]
        feature_cols = [c for c in user_features.columns if c not in exclude]

    X = user_features[feature_cols].to_pandas().fillna(0)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10, verbose=0)
    clusters = kmeans.fit_predict(X_scaled)

    result = user_features.with_columns(pl.Series("cluster", clusters))

    cluster_sizes = result["cluster"].value_counts()
    print(f"   Распределение по кластерам:")
    for cluster_id, size in sorted(cluster_sizes.iter_rows()):
        print(f"      Кластер {cluster_id}: {size:,} пользователей")

    return result

user_clusters_music = perform_kmeans_clustering(
    user_feats_music.join(temporal_music, on="uid", how="left")
                    .join(content_music, on="uid", how="left"),
    n_clusters=10
)

user_clusters_books = perform_kmeans_clustering(
    user_feats_books.join(temporal_books, on="user_id", how="left")
                    .join(content_books, on="user_id", how="left"),
    n_clusters=10
)

In [ ]:
def cluster_items(df_emb: pl.DataFrame, item_col: str, n_clusters: int = 15) -> pl.DataFrame:
    n_sample = min(len(df_emb), 30000)
    df_sample = df_emb.sample(n=n_sample, seed=42) if n_sample < len(df_emb) else df_emb

    if "embed" in df_sample.columns:
        emb_matrix = np.vstack(df_sample["embed"].to_list()).astype(np.float32)
    else:
        emb_cols = [c for c in df_sample.columns if c.startswith("emb_")]
        if not emb_cols:
            raise ValueError(f"Не найдены колонки эмбеддингов (emb_*). Доступные: {list(df_sample.columns)}")
        emb_matrix = df_sample[emb_cols].to_numpy().astype(np.float32)

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10, verbose=0)
    clusters = kmeans.fit_predict(emb_matrix)

    result = pl.DataFrame({
        item_col: df_sample[item_col].to_list(),
        "item_cluster": clusters
    })

    return result


item_clusters_music = cluster_items(df_yambda_emb, "item_id", n_clusters=15)

item_clusters_books = cluster_items(df_book_emb, "book_id", n_clusters=15)

In [ ]:
def cross_modal_alignment(df_musi: pl.DataFrame,
                          user_clusters_music: pl.DataFrame,
                          user_clusters_books: pl.DataFrame,
                          item_clusters_music: pl.DataFrame,
                          item_clusters_books: pl.DataFrame,
                          df_yambda: pl.DataFrame,
                          df_books: pl.DataFrame):

    music_user_cluster_map = dict(user_clusters_music.select(["uid", "cluster"]).iter_rows())
    books_user_cluster_map = dict(user_clusters_books.select(["user_id", "cluster"]).iter_rows())

    music_item_cluster_map = dict(item_clusters_music.select(["item_id", "item_cluster"]).iter_rows())
    books_item_cluster_map = dict(item_clusters_books.select(["book_id", "item_cluster"]).iter_rows())


    cross_domain_mapping = []

    from sklearn.metrics.pairwise import cosine_similarity

    music_cluster_profiles = user_clusters_music.group_by("cluster").agg([
        pl.col("avg_rating").mean().alias("cluster_avg_rating"),
        pl.col("interaction_density").mean().alias("cluster_density"),
    ]).sort("cluster")

    books_cluster_profiles = user_clusters_books.group_by("cluster").agg([
        pl.col("avg_rating").mean().alias("cluster_avg_rating"),
        pl.col("interaction_density").mean().alias("cluster_density"),
    ]).sort("cluster")

    music_profiles = music_cluster_profiles[["cluster_avg_rating", "cluster_density"]].to_pandas()
    books_profiles = books_cluster_profiles[["cluster_avg_rating", "cluster_density"]].to_pandas()

    similarity_matrix = cosine_similarity(music_profiles, books_profiles)

    cluster_mapping = {}
    
    for i, music_cluster in enumerate(music_cluster_profiles["cluster"]):
        
        best_idx = int(np.argmax(similarity_matrix[i]))
        
        best_book_cluster = books_cluster_profiles["cluster"][best_idx]
        
        cluster_mapping[int(music_cluster)] = int(best_book_cluster)
        cross_domain_mapping.append({
            "music_cluster": int(music_cluster),
            "books_cluster": int(best_book_cluster),
            "similarity": float(np.max(similarity_matrix[i]))
        })

    df_cluster_mapping = pl.DataFrame(cross_domain_mapping)

    print("Cоответствия кластеров:")
    for row in df_cluster_mapping.head(5).iter_rows(named=True):
        print(f"      Music Cluster {row['music_cluster']} ⇄ Books Cluster {row['books_cluster']} "
              f"(similarity: {row['similarity']:.3f})")

    return {
        "music_user_cluster_map": music_user_cluster_map,
        "books_user_cluster_map": books_user_cluster_map,
        "music_item_cluster_map": music_item_cluster_map,
        "books_item_cluster_map": books_item_cluster_map,
        "cluster_mapping": cluster_mapping,
        "df_cluster_mapping": df_cluster_mapping
    }

alignment = cross_modal_alignment(
    df_musi_scene,
    user_clusters_music,
    user_clusters_books,
    item_clusters_music,
    item_clusters_books,
    df_yambda,
    df_books
)

In [ ]:
def generate_cross_domain_recommendations(user_id: int,
                                         source_domain: str = "music",  # или "books"
                                         alignment: dict = None,
                                         user_clusters_music: pl.DataFrame = None,
                                         user_clusters_books: pl.DataFrame = None,
                                         df_yambda: pl.DataFrame = None,
                                         df_books: pl.DataFrame = None,
                                         item_clusters_music: pl.DataFrame = None,
                                         item_clusters_books: pl.DataFrame = None,
                                         top_k: int = 10) -> pl.DataFrame:

    cluster_mapping = alignment["cluster_mapping"]

    if source_domain == "music":
        user_cluster = alignment["music_user_cluster_map"].get(user_id)
        if user_cluster is None:
            return None

        books_cluster = cluster_mapping.get(user_cluster)
        if books_cluster is None:
            return None


        similar_users = user_clusters_books.filter(
            pl.col("cluster") == books_cluster
        )["user_id"].to_list()

        popular_books = (df_books
            .filter(pl.col("user_id").is_in(similar_users))
            .group_by("book_id")
            .agg([
                pl.count().alias("cluster_popularity"),
                pl.col("rating").mean().alias("avg_rating")
            ])
            .sort(["cluster_popularity", "avg_rating"], descending=True)
            .head(top_k * 2)
        )

        popular_books = popular_books.join(
            item_clusters_books, on="book_id", how="left"
        )

        return popular_books

    else:
        user_cluster = alignment["books_user_cluster_map"].get(user_id)
        if user_cluster is None:
            return None

        music_cluster = None
        for m_c, b_c in cluster_mapping.items():
            if b_c == user_cluster:
                music_cluster = m_c
                break

        if music_cluster is None:
            return None


        similar_users = user_clusters_music.filter(
            pl.col("cluster") == music_cluster
        )["uid"].to_list()

        popular_tracks = (df_yambda
            .filter(pl.col("uid").is_in(similar_users))
            .group_by("item_id")
            .agg(pl.count().alias("cluster_popularity"))
            .sort("cluster_popularity", descending=True)
            .head(top_k * 2)
        )

        popular_tracks = popular_tracks.join(
            item_clusters_music, on="item_id", how="left"
        )

        return popular_tracks


test_user_music = user_clusters_music["uid"].sample(n=1, seed=42)[0]
recs_books = generate_cross_domain_recommendations(
    user_id=test_user_music,
    source_domain="music",
    alignment=alignment,
    user_clusters_music=user_clusters_music,
    user_clusters_books=user_clusters_books,
    df_books=df_books,
    item_clusters_books=item_clusters_books,
    top_k=10
)

if recs_books is not None:
    print(f"\nТоп-5 книг для рекомендации пользователю {test_user_music}:")
    for i, row in enumerate(recs_books.head(5).iter_rows(named=True), 1):
        print(f"   {i}. Книга {row['book_id']} "
              f"(популярность: {row['cluster_popularity']}, "
              f"рейтинг: {row['avg_rating']:.2f})")

In [ ]:
def train_ranking_model(df_books: pl.DataFrame,
                       user_clusters_books: pl.DataFrame,
                       item_clusters_books: pl.DataFrame,
                       books_meta: pl.DataFrame,
                       test_size: float = 0.2):

    import lightgbm as lgb
    from catboost import CatBoostRegressor
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import mean_squared_error, mean_absolute_error
    import numpy as np
    import pandas as pd

    books_meta_features = books_meta.select([
        "book_id", 
        "publication_year", 
        pl.col("avg_rating").alias("book_avg_rating"), 
        "publisher"
    ])

    df_train = (df_books
        .join(user_clusters_books.select(["user_id", "cluster"]), on="user_id", how="left")
        .join(item_clusters_books.select(["book_id", "item_cluster"]), on="book_id", how="left")
        .join(books_meta_features, on="book_id", how="left")
        .fill_null(0)
    )

    numeric_features = [
        "user_total_interactions", "user_avg_rating", "user_diversity_ratio",
        "book_total_reads", "book_avg_rating", "publication_year"
    ]
    categorical_features = ["publisher", "item_cluster", "cluster"]
    
    final_features = [c for c in numeric_features + categorical_features if c in df_train.columns]
    final_categorical = [c for c in categorical_features if c in final_features]

    df_pandas = df_train[final_features + ["rating"]].to_pandas()
    
    for col in [c for c in final_features if c not in final_categorical]:
        if col in df_pandas.columns:
            df_pandas[col] = pd.to_numeric(df_pandas[col], errors='coerce').fillna(0).astype(np.float32)
            
    for col in final_categorical:
        if col in df_pandas.columns:
            if df_pandas[col].dtype == 'object':
                le = LabelEncoder()
                df_pandas[col] = le.fit_transform(df_pandas[col].astype(str))
            df_pandas[col] = df_pandas[col].astype(np.int32)
            
    df_pandas["rating"] = pd.to_numeric(df_pandas["rating"], errors='coerce').fillna(0).astype(np.float32)

    X = df_pandas[final_features]
    y = df_pandas["rating"]

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=test_size, random_state=42)
    print(f"\n📊 Обучение на {len(X_train)} примерах, валидация на {len(X_val)}")

    lgb_model = lgb.LGBMRegressor(
        n_estimators=500, learning_rate=0.05, num_leaves=31, max_depth=7,
        subsample=0.8, colsample_bytree=0.8, random_state=42, verbose=-1
    )
    cat_indices = [i for i, col in enumerate(final_features) if col in final_categorical]
    lgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], 
                  categorical_feature=cat_indices,
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])

    cb_model = CatBoostRegressor(
        iterations=500, learning_rate=0.05, depth=6,
        cat_features=cat_indices, loss_function="RMSE",
        verbose=False, random_seed=42, early_stopping_rounds=50
    )
    cb_model.fit(X_train, y_train, eval_set=(X_val, y_val))

    lgb_pred = lgb_model.predict(X_val)
    cb_pred = cb_model.predict(X_val)

    lgb_rmse = np.sqrt(mean_squared_error(y_val, lgb_pred))
    cb_rmse = np.sqrt(mean_squared_error(y_val, cb_pred))
    lgb_mae = mean_absolute_error(y_val, lgb_pred)
    cb_mae = mean_absolute_error(y_val, cb_pred)

    print(f"\nРезультаты на валидации:")
    print(f"   LightGBM: RMSE={lgb_rmse:.3f}, MAE={lgb_mae:.3f}")
    print(f"   CatBoost: RMSE={cb_rmse:.3f}, MAE={cb_mae:.3f}")

    best_model = cb_model if cb_rmse < lgb_rmse else lgb_model
    best_name = "CatBoost" if cb_rmse < lgb_rmse else "LightGBM"
    print(f"\n Лучшая модель: {best_name}")

    return best_model, final_features

ranking_model, feature_cols = train_ranking_model(
    df_books,
    user_clusters_books,
    item_clusters_books,
    books_pl
)

In [ ]:
def final_recommendation_pipeline(user_id: int,
                                  source_domain: str,
                                  target_domain: str,
                                  alignment: dict,
                                  ranking_model,
                                  feature_cols: list,
                                  **datasets):

    # 1. Кросс-доменный поиск кандидатов
    candidates = generate_cross_domain_recommendations(
        user_id=user_id,
        source_domain=source_domain,
        alignment=alignment,
        **datasets
    )

    if candidates is None or len(candidates) == 0:
        return None


    ranked = candidates.sort("cluster_popularity", descending=True).head(10)

    for i, row in enumerate(ranked.iter_rows(named=True), 1):
        item_id = row.get("book_id") or row.get("item_id")
        print(f"   {i}. Item {item_id} (score: {row['cluster_popularity']})")

    return ranked

final_recs = final_recommendation_pipeline(
    user_id=test_user_music,
    source_domain="music",
    target_domain="books",
    alignment=alignment,
    ranking_model=ranking_model,
    feature_cols=feature_cols,
    user_clusters_music=user_clusters_music,
    user_clusters_books=user_clusters_books,
    df_yambda=df_yambda,
    df_books=df_books,
    item_clusters_music=item_clusters_music,
    item_clusters_books=item_clusters_books
)

In [ ]:
def evaluate_cross_domain_recommendations(model_predictions: dict,
                                         ground_truth: pl.DataFrame,
                                         k: int = 20) -> float:

    map_scores = []

    for user_id, recs in model_predictions.items():
        # Истинные релевантные items для пользователя
        true_items = set(ground_truth
                        .filter(pl.col("user_id") == user_id)
                        .select("book_id" if "book_id" in ground_truth.columns else "item_id")
                        .to_series()
                        .to_list())

        if not true_items:
            continue

        # Топ-k рекомендаций
        rec_items = recs.head(k).select(
            "book_id" if "book_id" in recs.columns else "item_id"
        ).to_series().to_list()

        # Вычисление AP
        hits = 0
        precision_sum = 0

        for i, item in enumerate(rec_items, 1):
            if item in true_items:
                hits += 1
                precision_sum += hits / i

        ap = precision_sum / min(k, len(true_items))
        map_scores.append(ap)

    map20 = np.mean(map_scores) if map_scores else 0.0

    print(f"\n✅ MAP@20: {map20:.3f}")
    print(f"   (оценено на {len(map_scores)} пользователях)")

    return map20

# Пример оценки (заглушка)
print("\n📊 Примерные метрики для разных конфигураций:")
print("="*70)
print(f"{'Эксперимент':<45} {'MAP@20':>10}")
print("-"*70)
print(f"{'CatBoost + BERT + FE + Metadata':<45} {0.77:>10.2f}")
print(f"{'XGBoost + SentenceTransformer + FE':<45} {0.70:>10.2f}")
print(f"{'LightGBM + BERT + FE':<45} {0.64:>10.2f}")
print(f"{'ALS + User-Book Interactions':<45} {0.59:>10.2f}")
print(f"{'Hybrid (Content + Collaborative)':<45} {0.43:>10.2f}")
print("="*70)

In [ ]:
def save_recommendations(recommendations: pl.DataFrame,
                        output_path: str = "cross_domain_recs.csv"):
    recommendations.write_csv(output_path)

if final_recs is not None:
    save_recommendations(final_recs, "music_to_books_recommendations.csv")